# 00 — Environment Setup & Project Introduction

**Study:** API-Only vs. MCP + ReAct Agent Architecture  
**Date:** March 2026

---

## Core Hypothesis

> *Strict, pre-defined API call chains work fine for linear agent workflows, but break down under dynamic, multi-hop reasoning. MCP + ReAct agents solve this by letting the model decide what to call and when — enabling adaptive workflows that APIs alone cannot support.*

## The Scenario

A customer service agent answers: **"Is my package going to arrive on time?"**

Steps required:
1. Look up the customer account
2. Find their active shipment(s) — one or many
3. Get weather at origin **and** destination
4. Check carrier status for delay flags
5. Synthesise a risk assessment with reasoning

**The edge case that tests the hypothesis:**  
Account `ACC-002` has **two shipments** to different cities, one carrier is disrupted, and one destination is under a blizzard warning. A rigid API chain will silently fail. An MCP + ReAct agent will handle it gracefully.

---

## Notebook Roadmap

| # | Notebook | What we learn |
|---|----------|---------------|
| 00 | This notebook | Setup, hypothesis, API primer |
| 01 | `01_api_strict_workflow` | Hard-coded chain — happy path then failure |
| 02 | `02_mcp_react_workflow` | MCP + ReAct — dynamic adaptation |
| 03 | `03_analysis_and_blog_prep` | Side-by-side comparison, charts, blog material |

## Step 1 — Install Dependencies

In [ ]:
# Run this once. After installing, restart the kernel.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "../requirements.txt", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
print(result.stderr[-1000:] if result.returncode != 0 else "✅ All packages installed.")

## Step 2 — Start Mock Servers

Open **two terminal windows** and run:

```bash
# Terminal 1 — Logistics API (port 8001)
cd /path/to/IntelligentShipmentAdvisor
uvicorn infrastructure.logistics_api:app --port 8001 --reload

# Terminal 2 — Weather API (port 8002)
uvicorn infrastructure.weather_api:app --port 8002 --reload
```

Or run the cell below to start them as background processes from within the notebook.

In [ ]:
import subprocess
import time
import sys
import os

# Change to project root so uvicorn can find the infrastructure module
os.chdir(os.path.abspath(".."))
print(f"Working directory: {os.getcwd()}")

logistics_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "infrastructure.logistics_api:app",
     "--port", "8001", "--log-level", "error"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

weather_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "infrastructure.weather_api:app",
     "--port", "8002", "--log-level", "error"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

time.sleep(2)  # give servers time to start
print(f"Logistics API PID: {logistics_proc.pid}")
print(f"Weather API PID:   {weather_proc.pid}")
print("Servers started. Run the health-check cell below.")

## Step 3 — Health Check All Endpoints

In [ ]:
import requests
from rich.console import Console
from rich.table import Table

console = Console()

CHECKS = [
    ("Logistics",  "http://localhost:8001/"),
    ("Logistics",  "http://localhost:8001/accounts/ACC-001"),
    ("Logistics",  "http://localhost:8001/accounts/ACC-002"),
    ("Logistics",  "http://localhost:8001/shipments?account_id=ACC-002"),
    ("Logistics",  "http://localhost:8001/carriers/CARR-B"),
    ("Weather",    "http://localhost:8002/"),
    ("Weather",    "http://localhost:8002/weather/Denver"),
    ("Weather",    "http://localhost:8002/weather/Miami"),
]

table = Table(title="Endpoint Health Check", show_lines=True)
table.add_column("Service",  style="cyan")
table.add_column("URL",      style="white")
table.add_column("Status",   style="bold")
table.add_column("Preview",  style="dim")

all_ok = True
for service, url in CHECKS:
    try:
        r = requests.get(url, timeout=5)
        status = f"[green]✅ {r.status_code}[/green]"
        preview = str(r.json())[:80]
    except Exception as e:
        status = f"[red]❌ ERROR[/red]"
        preview = str(e)[:80]
        all_ok = False
    table.add_row(service, url, status, preview)

console.print(table)
if all_ok:
    console.print("[bold green]All endpoints healthy. Ready to proceed to Notebook 01.[/bold green]")
else:
    console.print("[bold red]Some endpoints failed. Make sure both servers are running.[/bold red]")

## Step 4 — Explore the Mock Data

Let's get familiar with the data before building agents.

In [ ]:
import json
import requests
from rich import print as rprint

# The edge-case account — this is what will break the rigid chain
acc = requests.get("http://localhost:8001/accounts/ACC-002").json()
shipments = requests.get("http://localhost:8001/shipments?account_id=ACC-002").json()

rprint("[bold cyan]Account:[/bold cyan]")
rprint(acc)
rprint(f"\n[bold cyan]Shipments ({len(shipments)} found):[/bold cyan]")
for s in shipments:
    rprint(s)

In [ ]:
# Check the disrupted carrier and high-risk weather
carrier_b = requests.get("http://localhost:8001/carriers/CARR-B").json()
denver_wx  = requests.get("http://localhost:8002/weather/Denver").json()

rprint("[bold red]Disrupted Carrier:[/bold red]")
rprint(carrier_b)
rprint("\n[bold red]High-Risk Destination Weather:[/bold red]")
rprint(denver_wx)

## Step 5 — What is MCP? (Conceptual Primer)

### REST API — Developer Drives the Workflow

```
Code:  call /accounts/{id}
         → call /shipments?account_id={id}    # assumes one result
              → call /weather/{destination}   # hard-coded: destination only
                   → call /carriers/{id}
                        → return result
```

The developer decided the sequence, the branching logic, and what to do with each result. The LLM (if any) only sees the final output.

### MCP + ReAct — Model Drives the Workflow

```
Model: "I need to check the account first."
  THOUGHT: I should look up the account to get context.
  ACTION:  get_account(account_id="ACC-002")
  OBSERVE: {name: "Bob Martinez", tier: "Standard", home_city: "New York"}

  THOUGHT: Now get shipments — there might be more than one.
  ACTION:  get_shipments(account_id="ACC-002")
  OBSERVE: [{SHP-101, dest: Miami}, {SHP-102, dest: Denver}]  ← 2 shipments!

  THOUGHT: Two shipments, two destinations. Check weather for both.
  ACTION:  get_weather(city="Miami")
  OBSERVE: {risk_level: "low"}
  ACTION:  get_weather(city="Denver")
  OBSERVE: {risk_level: "high", condition: "Blizzard"}

  THOUGHT: Denver is high-risk. Check SHP-102's carrier.
  ACTION:  get_carrier_status(carrier_id="CARR-B")
  OBSERVE: {status: "disrupted", delay_hours: 48, note: "I-70 closed..."}

  THOUGHT: Both carrier AND weather confirm Denver package is at risk.
  FINAL:   "SHP-101 (Miami) is on track. SHP-102 (Denver) is HIGH RISK..."
```

**The model adapted dynamically.** No developer wrote `if len(shipments) == 2`. The MCP tool catalogue + ReAct loop gave the model the ability to reason and act iteratively.

---

## Next: Notebook 01 — `01_api_strict_workflow.ipynb`

We'll build the rigid API chain, watch it work on the happy path, then watch it fail on the edge case.

In [ ]:
# Optional: Set your LLM API key for notebooks 02 and 03
# You can also put these in a .env file at the project root.

import os
# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["LLM_PROVIDER"]      = "openai"  # or "anthropic"

print("LLM_PROVIDER:",       os.getenv("LLM_PROVIDER", "(not set)"))
print("OPENAI_API_KEY:",     "(set)" if os.getenv("OPENAI_API_KEY") else "(not set)")
print("ANTHROPIC_API_KEY:",  "(set)" if os.getenv("ANTHROPIC_API_KEY") else "(not set)")